# timmによるモデル呼び出しと事前学習モデルの利用

---
## 目的
`timm`（PyTorch Image Models）ライブラリを用いて，ResNet50・EfficientNetなど代表的な画像分類モデルを呼び出す方法と，事前学習済み重みを利用した推論の方法について理解する．また，`torchvision.models`（`torchvision_models.ipynb`）との違いについても触れる．

## モジュールのインポート
`timm`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q timm

import os
import urllib.request

import torch
import torch.nn.functional as F
import timm
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## timmとは
`timm`（PyTorch Image Models）は，Ross Wightman氏を中心に開発されている，PyTorch用の画像分類モデル集です．torchvisionと同様に代表的なモデルを提供していますが，次のような特徴があります．

- ResNetやEfficientNetなどの定番モデルに加え，学術論文で発表された最新のモデルや，学習方法（レシピ）違いの重みなど，非常に多くの種類が公開されている．
- `timm.create_model()`という単一の関数で，モデルの種類によらず統一的にモデルを作成できる．
- Hugging Face Hubと連携しており，多くの事前学習済み重みがHugging Face Hub経由で配布されている．

利用可能なモデルの一覧は`timm.list_models()`で確認でき，ワイルドカード（`*`）を使った名前の絞り込みもできます．

In [ ]:
all_models = timm.list_models(pretrained=True)
print('事前学習済み重みが利用可能なモデル数:', len(all_models))

resnet_models = timm.list_models('resnet50*', pretrained=True)
print('resnet50系の事前学習済みモデル:', resnet_models[:10])

## モデルの作成と事前学習済み重みの読み込み
`timm.create_model(model_name, pretrained=True)`とすることで，指定したモデルに対応する事前学習済み重みが自動的にダウンロード・読み込みされます．torchvisionのように，モデルごとに異なる`<モデル名>_Weights`列挙型を指定する必要がなく，どのモデルでも同じ書き方で呼び出せる点が特徴です．

In [ ]:
model = timm.create_model('resnet50', pretrained=True)
model = model.to(device)
model.eval()  # 事前学習モデルを推論に使う際は必ずevalモードにする

print(type(model))

## 前処理の設定
timmのモデルは，torchvisionのようにモデルへ前処理を紐づけた`weights.transforms()`を持ちません．代わりに，モデルが保持する学習時の設定（`model.pretrained_cfg`）から，`timm.data.resolve_data_config()`と`timm.data.create_transform()`を用いて，対応する前処理を組み立てます．

In [ ]:
config = timm.data.resolve_data_config(model.pretrained_cfg)
preprocess = timm.data.create_transform(**config)
print(config)
print(preprocess)

## サンプル画像・クラス名の準備
推論に使用するサンプル画像と，ImageNetの1000クラスのクラス名一覧をダウンロードします．torchvisionの`Weights.meta['categories']`とは異なり，timmはクラス名の一覧を直接は保持していないため，別途用意する必要があります．

In [ ]:
img_path = 'dog.jpg'
if not os.path.exists(img_path):
    urllib.request.urlretrieve('https://github.com/pytorch/hub/raw/master/images/dog.jpg', img_path)
image = Image.open(img_path).convert('RGB')

classes_path = 'imagenet_classes.txt'
if not os.path.exists(classes_path):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt', classes_path)
with open(classes_path) as f:
    categories = [line.strip() for line in f.readlines()]

plt.imshow(image)
plt.axis('off')
plt.show()

## ResNet50による推論

In [ ]:
input_tensor = preprocess(image).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(input_tensor)
probs = F.softmax(output[0], dim=0)

top5_prob, top5_idx = torch.topk(probs, 5)
print('ResNet50の予測結果（上位5クラス）')
for prob, idx in zip(top5_prob, top5_idx):
    print(f'  {categories[idx]}: {prob.item():.4f}')

## EfficientNetによる推論
モデルの種類が変わっても手順は共通です．モデルごとに前処理の設定（入力サイズや正規化パラメータ）が異なる場合があるため，モデルを切り替えるたびに`resolve_data_config`・`create_transform`をやり直す必要がある点に注意してください．

In [ ]:
model_effnet = timm.create_model('efficientnet_b0', pretrained=True)
model_effnet = model_effnet.to(device)
model_effnet.eval()

config_effnet = timm.data.resolve_data_config(model_effnet.pretrained_cfg)
preprocess_effnet = timm.data.create_transform(**config_effnet)

input_tensor_effnet = preprocess_effnet(image).unsqueeze(0).to(device)

with torch.no_grad():
    output_effnet = model_effnet(input_tensor_effnet)
probs_effnet = F.softmax(output_effnet[0], dim=0)

top5_prob_effnet, top5_idx_effnet = torch.topk(probs_effnet, 5)
print('EfficientNet-B0の予測結果（上位5クラス）')
for prob, idx in zip(top5_prob_effnet, top5_idx_effnet):
    print(f'  {categories[idx]}: {prob.item():.4f}')

## 分類層の確認と特徴抽出器としての利用
timmのモデルは，どのアーキテクチャでも`model.get_classifier()`で最終の分類層を取得でき，`model.reset_classifier(num_classes)`で分類層のクラス数を変更できます．また，`num_classes=0`を指定してモデルを作成すると，分類層を持たない特徴抽出器として利用でき，Global Average Pooling後の特徴ベクトルがそのまま出力されます．これは，次のノートブック（`transfer_learning.ipynb`）で行うファインチューニングの土台となる仕組みです．

In [ ]:
print('ResNet50の分類層:', model.get_classifier())

feature_extractor = timm.create_model('resnet50', pretrained=True, num_classes=0)
feature_extractor = feature_extractor.to(device).eval()

with torch.no_grad():
    features = feature_extractor(input_tensor)
print('特徴ベクトルの形状:', features.shape)

## torchvisionとtimmの違い

| | torchvision.models | timm |
|---|---|---|
| モデルの呼び出し方 | モデルごとの専用関数（`resnet50()`など） | `create_model(model_name)`に統一 |
| 事前学習済み重みの指定 | モデルごとの`<モデル名>_Weights`列挙型 | `pretrained=True`のみで指定可能 |
| 前処理の取得 | `weights.transforms()` | `resolve_data_config` + `create_transform` |
| クラス名の取得 | `weights.meta['categories']`（付属） | 付属せず別途用意が必要 |
| 収録モデル数・種類の豊富さ | 標準的な代表モデルが中心 | 非常に多くの最新モデルを収録 |

用途に応じて，PyTorch公式がメンテナンスする安定性を重視するならtorchvision，最新のモデルや豊富な選択肢を利用したいならtimm，という使い分けが考えられます．

## 課題

1. `timm.list_models(pretrained=True)`で使用可能な事前学習済みモデルの総数を確認し，torchvisionで使用可能なモデル数（`torchvision.models.list_models()`）と比較しましょう．

2. 他のモデル（例：`vit_base_patch16_224`, `convnext_tiny`）をtimmで読み込み，同じ画像に対する予測結果を比較しましょう．

3. `timm.list_models('efficientnet*', pretrained=True)`でEfficientNet系のモデル一覧を確認し，いくつか異なるバリエーション（`efficientnet_b0`, `efficientnet_b3`など）で推論結果とモデルサイズを比較しましょう．